# 12 — Datos del mapa de validación (app web)
Prepara los datos de la app web **mapa-validacion** (Databricks Apps). La app muestra sobre un mapa de los 6 distritos de estudio lo que hay en el lakehouse, para revisarlo a ojo:
- distritos (`silver_distritos`) y hexágonos H3 r8 con los edificios (`silver_edificios`) y el riesgo de cada escenario (`riesgo_h3`, `riesgo_sa_roca_h3`, `riesgo_escenarios`);
- todos los edificios con su huella, la validación de pisos con OSM (`bronze_osm_niveles`), la grilla de suelo (`silver_suelo`) y las estaciones CISMID (`silver_ondas_registros`);
- chequeos automáticos: H3 recalculado, distrito por point-in-polygon y centroide vs. huella.

El resultado es un JSON en el Volume `web` (`web/mapa_validacion/datos.json`). La app lo lee y, cuando el job lo vuelve a escribir, se actualiza sola.

## 1. Setup — imports

In [ ]:
%pip install -q shapely

In [ ]:
import json
import math
import os
import time

import numpy as np
import pandas as pd
import shapely

## 2. Catálogo, esquema y parámetros

In [ ]:
catalogo = "grupo06proyectofinal"
esquema = "sismos"
volumen = f"/Volumes/{catalogo}/{esquema}"

spark.sql(f"USE CATALOG {catalogo}")
spark.sql(f"USE SCHEMA {esquema}")
spark.conf.set("spark.sql.session.timeZone", "America/Lima")

In [ ]:
spark.sql("CREATE VOLUME IF NOT EXISTS web")
CARPETA_WEB = f"{volumen}/web/mapa_validacion"
ARCHIVO_DATOS = f"{CARPETA_WEB}/datos.json"
os.makedirs(CARPETA_WEB, exist_ok=True)
E = "silver_edificios"
t0 = time.perf_counter()
print("Destino de los datos de la app:", ARCHIVO_DATOS)

## 3. Funciones

In [ ]:
def tabla(sql):
    return spark.sql(sql).toPandas()


def redondo(v, decimales):
    return None if v is None or (isinstance(v, float) and math.isnan(v)) else round(float(v), decimales)


def columnas(df, redondeo=None):
    """DataFrame -> dict de listas (formato columnar, NaN -> None) para el JSON."""
    redondeo = redondeo or {}
    out = {}
    for c in df.columns:
        s = df[c]
        if pd.api.types.is_bool_dtype(s) or (s.dtype == object and s.notna().any() and s.dropna().map(lambda v: isinstance(v, (bool, np.bool_))).all()):
            out[c] = [None if pd.isna(v) else int(bool(v)) for v in s]
        elif pd.api.types.is_numeric_dtype(s):
            a = s.to_numpy(dtype="float64", na_value=np.nan)
            if c in redondeo:
                a = np.round(a, redondeo[c])
            out[c] = [None if math.isnan(v) else (int(v) if float(v).is_integer() else float(v)) for v in a]
        else:
            out[c] = [None if pd.isna(v) else str(v) for v in s]
    return out


def categorias(df, cols):
    """Reemplaza el texto repetido por índices; devuelve los diccionarios."""
    dic = {}
    for c in cols:
        cats = sorted(df[c].dropna().astype(str).unique().tolist())
        idx = {v: i for i, v in enumerate(cats)}
        df[c] = pd.array([None if pd.isna(v) else idx[str(v)] for v in df[c]], dtype="Int64")
        dic[c] = cats
    return dic

## 4. Chequeos globales sobre todos los edificios

In [ ]:
r = spark.sql(f"""
SELECT count(*) AS total,
       count_if(h3_r8 IS NULL) AS sin_h3_r8,
       count_if(h3_r8 <> h3_longlatash3string(lon, lat, 8)) AS h3_r8_distinto,
       count_if(h3_resolution(h3_id) <> 9) AS h3_id_no_r9,
       count_if(h3_id <> h3_longlatash3string(lon, lat, 9)) AS h3_r9_distinto,
       count_if(h3_toparent(h3_id, 8) <> h3_r8) AS padre_distinto,
       count_if(distrito IS NULL) AS sin_distrito,
       count_if(altura_imputada) AS altura_imputada,
       count_if(altura_m IS NULL) AS sin_altura,
       count_if(pisos IS NULL) AS sin_pisos,
       count_if(vs30 IS NULL) AS sin_vs30,
       count_if(Tb_TP IS NULL) AS sin_Tb_TP,
       count_if(altura_m / pisos > 6 OR altura_m / pisos < 1.8) AS altura_por_piso_rara,
       min(lon) AS lon_min, max(lon) AS lon_max, min(lat) AS lat_min, max(lat) AS lat_max,
       percentile_approx(altura_m, array(0.01, 0.5, 0.99)) AS altura_p01_p50_p99,
       max(altura_m) AS altura_max, max(pisos) AS pisos_max
FROM {E}""").first().asDict()
chequeos_globales = {k: (list(v) if isinstance(v, (list, tuple)) else v) for k, v in r.items()}
for nombre, valor in chequeos_globales.items():
    print(f"  {nombre:<22} {valor}")

## 5. Distritos de estudio

In [ ]:
dist = tabla(f"""
SELECT d.ubigeo, d.distrito, d.provincia, d.area_km2, d.geometria_fuente, d.geometry_wkt, e.edificios
FROM silver_distritos d
LEFT JOIN (SELECT distrito, count(*) AS edificios FROM {E} GROUP BY distrito) e USING (distrito)
WHERE d.en_estudio""")
geoms_dist = shapely.from_wkt(dist.geometry_wkt.to_numpy())
features_dist = []
for fila, g in zip(dist.itertuples(), geoms_dist):
    if g is None:
        continue
    gs = shapely.set_precision(shapely.simplify(g, 0.00003, preserve_topology=True), 0.000001)
    features_dist.append({"type": "Feature", "geometry": json.loads(shapely.to_geojson(gs)),
                          "properties": {"distrito": fila.distrito, "provincia": fila.provincia, "ubigeo": fila.ubigeo,
                                         "area_km2": redondo(fila.area_km2, 2), "fuente": fila.geometria_fuente,
                                         "edificios": int(fila.edificios) if pd.notna(fila.edificios) else 0}})
print("Distritos:", len(features_dist))

## 6. Hexágonos H3 r8 y riesgo por escenario

In [ ]:
hex_base = tabla(f"""
SELECT h3_r8, count(*) AS edificios, avg(pisos) AS pisos_medio, max(pisos) AS pisos_max, avg(altura_m) AS altura_media,
       avg(CAST(altura_imputada AS INT)) * 100 AS pct_imputada, avg(vs30) AS vs30_medio, avg(Tb_TP) AS Tb_TP_medio,
       mode(tipo_suelo) AS tipo_suelo, mode(distrito) AS distrito, count(DISTINCT distrito) AS n_distritos,
       h3_boundaryasgeojson(h3_r8) AS borde
FROM {E} WHERE h3_r8 IS NOT NULL GROUP BY h3_r8""")
riesgo = tabla("""SELECT h3_r8, escenario, edificios, Sa_Tb_mediana_g, P_colapso_media, dano_esperado_medio,
                         pct_P_colapso_mayor_umbral, pct_resonancia, pct_fuera_cobertura FROM riesgo_h3""")
sa_roca = tabla("SELECT h3_r8, escenario, pga_roca_g, kriging_sd_rel, dist_estacion_km FROM riesgo_sa_roca_h3")
escenarios = tabla("SELECT escenario, evento, tipo, pga_roca_escenario_g FROM riesgo_escenarios ORDER BY tipo, escenario")

METRICAS_ESC = ["Sa_Tb_mediana_g", "P_colapso_media", "dano_esperado_medio", "pct_P_colapso_mayor_umbral",
                "pct_resonancia", "pct_fuera_cobertura", "pga_roca_g", "kriging_sd_rel", "dist_estacion_km"]
riesgo = riesgo.merge(sa_roca, on=["h3_r8", "escenario"], how="left")
idx_hex = {h: i for i, h in enumerate(hex_base.h3_r8)}
por_esc = {}
for esc, d in riesgo.groupby("escenario"):
    pos = d.h3_r8.map(idx_hex)
    por_esc[esc] = {}
    for m in METRICAS_ESC:
        a = [None] * len(hex_base)
        for p, v in zip(pos, d[m]):
            if pd.notna(p) and pd.notna(v):
                a[int(p)] = round(float(v), 5)
        por_esc[esc][m] = a
hex_sin_edificios = sorted(set(riesgo.h3_r8) - set(hex_base.h3_r8))
edif_riesgo = riesgo.groupby("escenario").edificios.sum().to_dict()

bordes = []
for b in hex_base.borde:
    anillo = json.loads(b)["coordinates"][0]
    bordes.append([[round(x, 6), round(y, 6)] for x, y in anillo[:-1]])
hex_out = columnas(hex_base.drop(columns=["borde"]),
                   {"pisos_medio": 2, "altura_media": 1, "pct_imputada": 1, "vs30_medio": 0, "Tb_TP_medio": 3})
hex_out["borde"] = bordes
print(f"Hexágonos con edificios: {len(hex_base)} | en riesgo_h3 sin edificios: {len(hex_sin_edificios)} | escenarios: {len(por_esc)}")

## 7. Pares OSM → edificio
Cada punto de OSM con `building:levels` se empareja con la huella que lo contiene (`st_contains`); si ninguna lo contiene, con el edificio de centroide más cercano a menos de 8 m. Se compara `pisos_25d` (la estimación 2.5D, sin OSM) para que no sea circular.

In [ ]:
spark.sql(f"""
WITH o AS (SELECT osm_id, lon, lat, niveles, explode(h3_kring(h3_longlatash3string(lon, lat, 9), 1)) AS celda
           FROM bronze_osm_niveles WHERE niveles BETWEEN 1 AND 40),
cand AS (SELECT o.osm_id, e.id, e.pisos_25d AS pisos, e.pisos AS pisos_final, e.pisos_metodo, e.altura_m, e.altura_metodo, e.area_m2,
                st_contains(st_geomfromwkt(e.geometry_wkt), st_point(o.lon, o.lat)) AS dentro,
                st_distance(st_transform(st_point(o.lon, o.lat, 4326), 32718), st_transform(st_point(e.lon, e.lat, 4326), 32718)) AS dist_m
         FROM o JOIN {E} e ON e.h3_id = o.celda),
mejor AS (SELECT * FROM (SELECT *, row_number() OVER (PARTITION BY osm_id ORDER BY dentro DESC, dist_m) AS r FROM cand)
          WHERE r = 1 AND (dentro OR dist_m < 8))
SELECT o.osm_id, o.lon, o.lat, o.niveles, m.id, m.pisos, m.pisos_final, m.pisos_metodo, m.altura_m, m.altura_metodo, m.area_m2, m.dist_m,
       CASE WHEN m.dentro THEN 'huella' WHEN m.id IS NOT NULL THEN 'centroide_8m' ELSE 'sin_pareja' END AS pareja
FROM (SELECT DISTINCT osm_id, lon, lat, niveles FROM bronze_osm_niveles WHERE niveles BETWEEN 1 AND 40) o
LEFT JOIN mejor m USING (osm_id)""").createOrReplaceTempView("pares_osm")
pares_osm = tabla("SELECT * FROM pares_osm")
print("Edificios de OSM con pisos:", len(pares_osm), "| con pareja:", int(pares_osm.id.notna().sum()))

## 8. Edificios con su huella y chequeos geométricos

In [ ]:
edif = tabla(f"""
SELECT id, distrito, h3_r8, lon, lat, area_m2, confianza, altura_m, altura_metodo, pisos, pisos_metodo, Tb, Tb_TP,
       vs30, tipo_suelo, perfil_fuente, zona_cismid, con_microzonificacion, pisos_25d, niveles_osm, altura_max_m, geometry_wkt
FROM {E}""")
print(f"Edificios: {len(edif):,}")

puntos = shapely.points(edif.lon.to_numpy(), edif.lat.to_numpy())
arbol = shapely.STRtree(geoms_dist)
i_pt, i_pol = arbol.query(puntos, predicate="within")
dist_pip = np.full(len(edif), None, dtype=object)
dist_pip[i_pt] = dist.distrito.to_numpy()[i_pol]
edif["distrito_pip"] = dist_pip
edif["distrito_ok"] = [a == b for a, b in zip(edif.distrito, edif.distrito_pip)]

huellas = shapely.from_wkt(edif.geometry_wkt.to_numpy())
edif["dist_centroide_m"] = np.round(
    np.hypot((shapely.get_x(shapely.centroid(huellas)) - edif.lon) * 111320 * np.cos(np.radians(edif.lat)),
             (shapely.get_y(shapely.centroid(huellas)) - edif.lat) * 110574), 1)
geo = []
for g, x0, y0 in zip(huellas, edif.lon.to_numpy(), edif.lat.to_numpy()):
    if g is None or g.is_empty:
        geo.append([])
        continue
    if g.geom_type == "MultiPolygon":
        g = max(g.geoms, key=lambda p: p.area)
    xy = np.asarray(g.exterior.coords)[:-1]
    geo.append(np.round((xy - [x0, y0]) * 1e6).astype(int).ravel().tolist())
edif = edif.drop(columns=["geometry_wkt", "distrito_pip"])
cats_edif = categorias(edif, ["distrito", "altura_metodo", "pisos_metodo", "tipo_suelo", "perfil_fuente"])
edif_out = columnas(edif, {"lon": 7, "lat": 7, "area_m2": 1, "confianza": 3, "altura_m": 1, "altura_max_m": 1, "Tb": 3, "Tb_TP": 3,
                            "vs30": 0, "niveles_osm": 1})
edif_out["g"] = geo
chequeos_muestra = {"muestra": len(edif), "distrito_distinto": int((~edif.distrito_ok).sum()),
                    "centroide_lejos_5m": int((edif.dist_centroide_m > 5).sum())}
print("Chequeos de los edificios:", chequeos_muestra)

## 9. Validación de pisos con OSM (dato real)

In [ ]:
osm = pares_osm.copy()
con_par = osm[osm.id.notna()]
dif = con_par.pisos - con_par.niveles
resumen_osm = {"n_osm": len(osm), "n": len(con_par), "por_huella": int((con_par.pareja == "huella").sum()),
               "mae": redondo(dif.abs().mean(), 2), "sesgo": redondo(dif.mean(), 2),
               "pct_exacto": redondo((dif == 0).mean() * 100, 1), "pct_pm1": redondo((dif.abs() <= 1).mean() * 100, 1),
               "corr": redondo(con_par[["pisos", "niveles"]].astype(float).corr().iloc[0, 1], 3)}
GRUPOS_OSM = [(1, 1, "1"), (2, 2, "2"), (3, 3, "3"), (4, 4, "4"), (5, 6, "5–6"), (7, 9, "7–9"), (10, 14, "10–14"), (15, 40, "15+")]
por_grupo = []
for a, b, nombre in GRUPOS_OSM:
    g = osm[osm.niveles.between(a, b)]
    p = g[g.id.notna()]
    d = p.pisos - p.niveles
    por_grupo.append({"grupo": nombre, "osm": len(g), "con_par": len(p), "niveles_medio": redondo(p.niveles.mean(), 1),
                      "pisos_medio": redondo(p.pisos.mean(), 1), "altura_media": redondo(p.altura_m.mean(), 1),
                      "mae": redondo(d.abs().mean(), 2), "sesgo": redondo(d.mean(), 2),
                      "pct_pm1": redondo((d.abs() <= 1).mean() * 100, 0)})
resumen_osm["por_grupo"] = por_grupo
peores = con_par[con_par.niveles >= 10].assign(err=lambda x: x.pisos - x.niveles).sort_values("err").head(40)
resumen_osm["con_pisos_osm"] = int((con_par.pisos_metodo == "osm").sum())
resumen_osm["peores"] = [{"osm_id": r.osm_id, "lon": r.lon, "lat": r.lat, "niveles": float(r.niveles), "pisos": int(r.pisos),
                          "pisos_final": int(r.pisos_final), "altura_m": redondo(r.altura_m, 1)} for r in peores.itertuples()]
cats_osm = categorias(osm, ["pisos_metodo", "altura_metodo", "pareja"])
osm_out = columnas(osm.drop(columns=["area_m2"]), {"lon": 7, "lat": 7, "dist_m": 1, "altura_m": 1})
print("Validación con OSM:", {k: v for k, v in resumen_osm.items() if k not in ("por_grupo", "peores")})
print(pd.DataFrame(por_grupo).to_string(index=False))

## 10. Suelo y estaciones

In [ ]:
suelo = tabla("SELECT lon, lat, vs30, vs30_usgs, relleno_mar, perfil, distrito, en_area_estudio FROM silver_suelo WHERE en_area_estudio")
paso = float(np.min(np.diff(np.sort(suelo.lon.unique()))))
cats_suelo = categorias(suelo, ["perfil", "distrito"])
suelo_out = columnas(suelo, {"lon": 6, "lat": 6, "vs30": 0, "vs30_usgs": 0})
est = tabla("""
SELECT estacion, first(estacion_nombre) AS nombre, first(red) AS red, avg(lat) AS lat, avg(lon) AS lon, first(vs30) AS vs30,
       count(*) AS registros, max(pga_geo_g) AS pga_max_g, concat_ws(', ', collect_set(evento)) AS eventos
FROM silver_ondas_registros GROUP BY estacion""")
est_out = columnas(est, {"lat": 5, "lon": 5, "vs30": 0, "pga_max_g": 4})
print(f"Celdas de suelo: {len(suelo):,} (paso {paso:.5f}°) | estaciones: {len(est)}")

## 11. Guardar los datos de la app en el Volume web

In [ ]:
datos = {
    "generado": pd.Timestamp.now(tz="America/Lima").strftime("%Y-%m-%d %H:%M") + " (hora de Lima)",
    "chequeos": {"globales": chequeos_globales, "muestra": chequeos_muestra, "osm": resumen_osm,
                 "hex_sin_edificios": len(hex_sin_edificios), "hex_con_edificios": len(hex_base),
                 "edificios_en_riesgo_h3": {k: int(v) for k, v in edif_riesgo.items()}},
    "distritos": {"type": "FeatureCollection", "features": features_dist},
    "hex": hex_out, "hex_esc": por_esc,
    "escenarios": columnas(escenarios, {"pga_roca_escenario_g": 3}),
    "edificios": edif_out, "cats_edif": cats_edif,
    "osm": osm_out, "cats_osm": cats_osm,
    "suelo": suelo_out, "cats_suelo": cats_suelo, "suelo_paso": paso,
    "estaciones": est_out,
}
texto = json.dumps(datos, ensure_ascii=False, separators=(",", ":"), allow_nan=False)
temporal = ARCHIVO_DATOS + ".tmp"
with open(temporal, "w", encoding="utf-8") as f:
    f.write(texto)
os.replace(temporal, ARCHIVO_DATOS)
mb = os.path.getsize(ARCHIVO_DATOS) / 1e6
print(f"Listo: {ARCHIVO_DATOS} ({mb:.1f} MB) en {time.perf_counter() - t0:.0f} s")
dbutils.notebook.exit(json.dumps({"archivo": ARCHIVO_DATOS, "mb": round(mb, 1), "edificios": len(edif), "hexagonos": len(hex_base)}))